In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install cudaq


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.9/156.9 MB 14.5 MB/s eta 0:00:00
  Created wheel for cudaq: filename=cudaq-0.9.1-py3-none-any.whl size=6787 sha256=6b3df41f3a6d0825dd5594a6052d4daf9ac18fe744f0668d09a3479b1a9cc5ef
  Stored in directory: /root/.cache/pip/wheels/b0/ee/32/0ac9053509c0ba31e4bac6122454996a9a4e4d6523b1918d04
Successfully built cudaq


In [ ]:
!pip install qutip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.1/30.1 MB 73.2 MB/s eta 0:00:00


#Now testing and preperations are done, we shall move to the real work


##Reading the files and applying one hot encoding

In [ ]:
import os
from collections import Counter

# Path to the folder
folder_path = "/content/drive/MyDrive/Project_health_hackathon-20241112T014818Z-001/Project_health_hackathon/FastQ_raw"

# Get all files in the folder
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

# Count total files
total_files = len(file_list)

# Extract file extensions
file_extensions = [os.path.splitext(f)[1] for f in file_list]

# Count file types
file_types_count = Counter(file_extensions)

# Print results
print(f"Total number of files: {total_files}")
print("File types and their counts:")
for ext, count in file_types_count.items():
    print(f"{ext if ext else 'No Extension'}: {count}")

Total number of files: 0
File types and their counts:


In [ ]:
!pip install transformers
!pip install tokenizers

#MODELS--Regression version





In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras import layers, models
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import gzip

Prepare the x y data.

In [ ]:
import os
import pandas as pd
import numpy as np
from tensorflow.keras import layers, models
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import gzip

# Placeholder functions for loading FASTQ sequences and one-hot encoding
def load_fastq_sequences(file_path, num_reads=1000):
    sequences = []
    with gzip.open(file_path, 'rt') as file:
        while len(sequences) < num_reads:
            file.readline()  # Read header
            sequence = file.readline().strip()  # Read sequence
            file.readline()  # Skip '+'
            file.readline()  # Skip quality score
            sequences.append(sequence)
    return sequences

def one_hot_encode_sequence(sequence, max_length=100):
    one_hot_mapping = {
        'A': [0, 0, 0, 1],
        'T': [0, 0, 1, 0],
        'G': [1, 0, 0, 0],
        'C': [0, 1, 0, 0]
    }
    encoded_sequence = [one_hot_mapping.get(nuc, [0, 0, 0, 0]) for nuc in sequence[:max_length]]
    if len(encoded_sequence) < max_length:
        encoded_sequence += [[0, 0, 0, 0]] * (max_length - len(encoded_sequence))
    return np.array(encoded_sequence)

# Path to the CSV file with metadata
csv_path = '/content/drive/MyDrive/health_hackathon/Project_health_hackathon/combined_metadata.csv'
sample_data = pd.read_csv(csv_path)

data = []  # Placeholder for one-hot encoded data
labels = []  # Placeholder for age labels

print("Processing Metadata and FASTQ Sequences:")

# Process each sample in the metadata
for index, sample in sample_data.iterrows():
    srr_id = sample.get("Run")
    age = sample.get("age")

    # Skip invalid samples
    if pd.isna(srr_id) or pd.isna(age):
        print(f"Skipping invalid sample at index {index}: {sample}")
        continue

    # Load sequences for single or paired-end files
    try:
        sequences = []

        # Check for paired-end files (_1 and _2)
        fastq_file_1 = f'/content/drive/MyDrive/health_hackathon/Project_health_hackathon/ENA_Downloads/{srr_id}_1.fastq.gz'
        fastq_file_2 = f'/content/drive/MyDrive/health_hackathon/Project_health_hackathon/ENA_Downloads/{srr_id}_2.fastq.gz'
        single_file = f'/content/drive/MyDrive/health_hackathon/Project_health_hackathon/ENA_Downloads/{srr_id}.fastq.gz'

        if os.path.exists(fastq_file_1) and os.path.exists(fastq_file_2):
            print(f"Processing paired-end files for {srr_id}...")
            seq_1 = load_fastq_sequences(fastq_file_1)
            seq_2 = load_fastq_sequences(fastq_file_2)
            sequences = [s1 + s2 for s1, s2 in zip(seq_1, seq_2)]  # Concatenate paired-end reads
        elif os.path.exists(single_file):
            print(f"Processing single-end file for {srr_id}...")
            sequences = load_fastq_sequences(single_file)
        else:
            print(f"No valid FASTQ file found for {srr_id}. Skipping.")
            continue

        # One-hot encode and flatten the sequences
        encoded_sequences = [one_hot_encode_sequence(seq) for seq in sequences]
        flattened_sequence = np.concatenate(encoded_sequences, axis=0)
        data.append(flattened_sequence)
        labels.append(age)
    except Exception as e:
        print(f"Error processing {srr_id}: {e}")
        continue

# Convert data and labels to NumPy arrays
if not data or not labels:
    raise ValueError("No valid data or labels found. Check metadata and FASTQ files.")
data = np.array(data)
labels = np.array(labels)

print(f"Data shape: {data.shape}")
print(f"Labels shape: {labels.shape}")

# Ensure the data shape is compatible with Conv1D
data = data.reshape(data.shape[0], -1, 4)

# Split into training, validation, and test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Test data shape: {X_test.shape}")

Processing Metadata and FASTQ Sequences:
Processing paired-end files for SRR23920268...
Processing paired-end files for SRR23920267...
Processing paired-end files for SRR23920266...
Processing paired-end files for SRR23920265...
Processing paired-end files for SRR23920264...
Processing paired-end files for SRR23920263...
Processing paired-end files for SRR23920262...
Processing paired-end files for SRR23920261...
Processing paired-end files for SRR23920260...
Processing paired-end files for SRR23920259...
Processing paired-end files for SRR23920258...
Processing paired-end files for SRR23920257...
Processing paired-end files for SRR23920256...
Processing paired-end files for SRR23920255...
Processing paired-end files for SRR23920254...
Processing paired-end files for SRR23920253...
Processing paired-end files for SRR23920252...
Processing paired-end files for SRR23920251...
Processing paired-end files for SRR23920250...
Processing paired-end files for SRR23920249...
Processing paired-e

##Test the k-mers method

In [ ]:
print(f"Training data: {X_train}")

print(y_train)
print(len(X_train))
print(len(y_train))

Training data: [[[1 0 0 0]
  [0 1 0 0]
  [0 0 1 0]
  ...
  [0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]]

 [[0 0 1 0]
  [0 0 0 0]
  [0 0 1 0]
  ...
  [0 1 0 0]
  [0 1 0 0]
  [1 0 0 0]]

 [[0 1 0 0]
  [0 0 0 0]
  [0 0 1 0]
  ...
  [0 1 0 0]
  [0 1 0 0]
  [0 0 1 0]]

 ...

 [[1 0 0 0]
  [0 0 0 1]
  [1 0 0 0]
  ...
  [0 0 0 1]
  [1 0 0 0]
  [0 0 0 1]]

 [[0 0 0 1]
  [0 0 0 0]
  [0 0 1 0]
  ...
  [1 0 0 0]
  [0 1 0 0]
  [0 0 0 1]]

 [[0 1 0 0]
  [0 0 1 0]
  [0 0 0 1]
  ...
  [0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]]]
[20 31 62 28 29 59 25 53 20 18 65 25 48 25 74 20 77 20 18 28 26 26 24 31
 36 27 51 50 31 62 20 26 37 30 43 59 51 51 42 65 41 35 35 51 58 30 29 62
 35 38 30 30 26 58 36 59 30 62 20 31 19 30 20 42 29 29 62 71 51 20 29 20
 28 58 70 51 31 22 20 27 51 20 20 20 71 45 53 20 57 30 20 28 51 56 38 36
 21 62 20 20 29 20 23 33 58 77 62 30 58 33 29 20 56 20 65 29 58 65 28 29
 20 31 35 20 48 25 30 20 36 54 26 23 40 20 25 85 23 42 20 20 56 20 30 51
 35 58 23 22 62 30 43 65 21 29 18 20 31 58 20 36 59 27 20 28 

#test the transformer method with k-mers

In [ ]:
import os
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import gzip
import torch
from torch.utils.data import Dataset

# First clone the DNABERT repository
!git clone https://github.com/jerryji1993/DNABERT.git
!mv DNABERT/ src/

# Install required dependencies
!pip install transformers
!pip install nvidia-pyindex
!pip install git+https://github.com/huggingface/transformers.git
!conda install pytorch torchvision cudatoolkit=10.0 -c pytorch

!cd DNABERT
!python3 -m pip install --editable .
!cd examples
!python3 -m pip install -r requirements.txt

Cloning into 'DNABERT'...
remote: Enumerating objects: 778, done.
remote: Counting objects: 100% (778/778), done.
remote: Compressing objects: 100% (348/348), done.
remote: Total 778 (delta 420), reused 721 (delta 407), pack-reused 0 (from 0)
Receiving objects: 100% (778/778), 11.66 MiB | 19.91 MiB/s, done.
Resolving deltas: 100% (420/420), done.
  Preparing metadata (setup.py) ... done
  Created wheel for nvidia-pyindex: filename=nvidia_pyindex-1.0.9-py3-none-any.whl size=8419 sha256=6a746e3873a7bd773d2891f6989297c1d44974132a8ff6f75a9813e3c8a8ac5a
  Stored in directory: /root/.cache/pip/wheels/49/d0/7d/b68b3665d16ee20355e65fb7ef48b7ca26533217d9f09924fe
Successfully built nvidia-pyindex
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-nmy4f1fj
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-nmy4f1fj
  Resolved https://github.com/huggingface/transformers.git to commit dd16acb8a3e93b6

/bin/bash: line 1: conda: command not found
/bin/bash: line 1: cd: DNABERT: No such file or directory
Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
/bin/bash: line 1: cd: examples: No such file or directory
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
# prompt: unzip this file /content/DNABERT/src/DNABERT/6-new-12w-0.zip

!unzip /content/DNABERT/src/DNABERT/6-new-12w-0.zip

unzip:  cannot find or open /content/DNABERT/src/DNABERT/6-new-12w-0.zip, /content/DNABERT/src/DNABERT/6-new-12w-0.zip.zip or /content/DNABERT/src/DNABERT/6-new-12w-0.zip.ZIP.


#Hypird Model

In [ ]:
pip install tensorcircuit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.0/342.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.1/244.1 kB 24.0 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade tensorcircuit  # Upgrade to the latest version

In [ ]:
pip install -U qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 11.6 MB/s eta 0:00:00


In [ ]:
pip install -U cirq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.7/532.7 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.5/596.5 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.9/202.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.0 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import tensorcircuit as tc

import logging
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold

In [ ]:
import tensorcircuit as tc
# ... other imports ...

# Set the default device for tensorcircuit to 'cuda'
tc.set_backend("pytorch")
tc.set_dtype("complex128")

('complex128', 'float64')

In [ ]:
class VariationalQuantumFeatureTransform(nn.Module):
    def __init__(self, input_dim=256, n_qubits=8, n_layers=7, device='cuda'):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.device = device  # ✅ Store device

        self.linear_in = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_qubits)
        ).to(self.device)  # ✅ Move to correct device

        self.linear_out = nn.Sequential(
            nn.Linear(n_qubits * 2, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        ).to(self.device)  # ✅ Move to correct device

        # Option: learnable scaling factor for the rotation angles.
        self.angle_scale = nn.Parameter(torch.tensor(20.2, device=self.device))  # ✅ Ensure tensor is on device

        # Increase q_weights initialization scale slightly if needed.
        self.q_weights = nn.Parameter(0.1 * torch.randn(n_layers, n_qubits, 3).to(self.device))  # ✅ Move to correct device

    def forward(self, x):
        # ✅ Ensure `x` is moved to `self.device`
        x = x.to(self.device)

        raw_x = self.linear_in(x)  # shape: (batch_size, n_qubits)
        print("Raw linear_in output: mean = {:.4f}, std = {:.4f}".format(raw_x.mean().item(), raw_x.std().item()))

        # Use tanh then scale to [-pi, pi] (or learnable scale)
        x = torch.tanh(raw_x) * self.angle_scale
        print("Scaled linear_in output (tanh): mean = {:.4f}, std = {:.4f}".format(x.mean().item(), x.std().item()))

        def quantum_net(data, weights):
            batch_size = data.shape[0]
            n_qubits = self.n_qubits
            expectations = []

            for b in range(batch_size):
                c = tc.Circuit(n_qubits)  # ✅ No `device` issue here

                # Enhanced encoding layer
                for j in range(n_qubits):
                    c.rx(j, theta=data[b, j].item())  # ✅ Ensure scalar input
                    c.ry(j, theta=data[b, (j+1) % n_qubits].item())

                # Improved variational layers
                for i in range(self.n_layers):
                    for j in range(n_qubits):
                        c.rz(j, theta=weights[i, j, 0].item())
                        c.ry(j, theta=weights[i, j, 1].item())
                        c.rz(j, theta=weights[i, j, 2].item())

                    # Adaptive entanglement
                    if i % 2 == 0:
                        for j in range(n_qubits - 1):
                            c.cnot(j, (j + 1) % n_qubits)
                    else:
                        for j in range(n_qubits):
                            for k in range(j + 1, n_qubits):
                                c.cnot(j, k)

                # Diverse measurements
                exp_vals = []
                for k in range(n_qubits):
                    exp_vals.append(torch.tensor(c.expectation_ps(z=[k]), device=self.device))
                    exp_vals.append(torch.tensor(c.expectation_ps(x=[k]), device=self.device))

                expectations.append(torch.stack(exp_vals))

            return torch.stack(expectations).to(self.device)  # ✅ Move results to correct device

        expectations = quantum_net(x, self.q_weights).real  # ✅ Ensure real values
        expectations = expectations.type(torch.float32)
        print("Quantum expectations: min = {:.4f}, max = {:.4f}".format(expectations.min().item(), expectations.max().item()))

        output = self.linear_out(expectations).squeeze()
        print("Final output (after linear_out):", output.detach().cpu().numpy())

        return output


In [ ]:
# ----------------------------------------------------------------------------
# Setup logging to both console and file
# ----------------------------------------------------------------------------
logging.basicConfig(
    filename="debug.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s"
)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s %(levelname)s: %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)

logging.info("Starting the script with debug logging...")

In [ ]:
import logging
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold

import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from torch.utils.data import DataLoader, TensorDataset

# ----------------------------------------------------------------------------
# Setup logging
# ----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

# ----------------------------------------------------------------------------
# Quantum Model Definition
# ----------------------------------------------------------------------------
class LargeRNAModelVQC(nn.Module):
    def __init__(self, input_dim=4, n_qubits=8, q_layers=7, device='cuda'):  # ✅ Added `device` to constructor
        super().__init__()
        self.device = device  # ✅ Store device

        self.downsample = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=16, stride=8),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(64, 128, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=4, stride=2),
            nn.AdaptiveAvgPool1d(256)
        )

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=512, batch_first=True),
            num_layers=4
        )

        self.q_regressor = VariationalQuantumFeatureTransform(n_qubits=n_qubits, n_layers=q_layers)
        self.pos_encoder = nn.Parameter(torch.randn(1, 256, 256).to(self.device))  # ✅ Moved to correct device

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (batch, 4, seq_len)
        x = self.downsample(x)  # (batch, 256, 256)
        x += self.pos_encoder
        x = x.permute(0, 2, 1)  # (batch, 256, 256)
        transformer_out = self.transformer(x)
        pooled = transformer_out.mean(dim=1)  # (batch, 256)
        return self.q_regressor(pooled)

# ----------------------------------------------------------------------------
# Training Utilities
# ----------------------------------------------------------------------------
def calculate_metrics(y_true, y_pred):
    return {
        'mae': torch.abs(y_true - y_pred).mean().item(),
        'mse': torch.mean((y_true - y_pred)**2).item(),
        'rmse': torch.sqrt(torch.mean((y_true - y_pred)**2)).item(),
        'r2': 1 - (torch.sum((y_true - y_pred)**2) / torch.sum((y_true - y_true.mean())**2)).item()
    }

def train_one_epoch(model, optimizer, criterion, dataloader, device):
    model.train()
    total_loss = 0

    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)  # ✅ Already on device
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(X_batch)

    return total_loss / len(dataloader.dataset)

def validate_model(model, criterion, dataloader, device):
    model.eval()
    val_loss = 0
    preds, trues = [], []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * len(X_batch)
            # Ensure outputs has at least one dimension before appending
            preds.append(outputs.cpu().unsqueeze(0) if outputs.dim() == 0 else outputs.cpu())
            trues.append(y_batch.cpu().unsqueeze(0) if y_batch.dim() == 0 else y_batch.cpu())

    preds, trues = torch.cat(preds), torch.cat(trues)
    metrics = calculate_metrics(trues, preds)
    return val_loss / len(dataloader.dataset), metrics

# ----------------------------------------------------------------------------
# Optuna Hyperparameter Tuning
# ----------------------------------------------------------------------------
def objective(trial):
    global X_train, y_train

    # Hyperparameter search space
    lr_classical = trial.suggest_float("lr_classical", 1e-5, 1e-3, log=True)
    lr_quantum = trial.suggest_float("lr_quantum", 1e-4, 1e-2, log=True)
    n_qubits = trial.suggest_int("n_qubits", 4, 16, step=4)
    q_layers = trial.suggest_int("q_layers", 2, 10)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)  # ✅ Change to 5 folds
    metrics_all_folds = {'mae': [], 'mse': [], 'rmse': [], 'r2': []}  # ✅ Store all metrics

    for train_idx, val_idx in kf.split(X_train):
        X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]

        # Convert to tensors
        X_train_fold = torch.tensor(X_train_fold, dtype=torch.float32).to(device)
        y_train_fold = torch.tensor(y_train_fold, dtype=torch.float32).to(device)
        X_val_fold = torch.tensor(X_val_fold, dtype=torch.float32).to(device)
        y_val_fold = torch.tensor(y_val_fold, dtype=torch.float32).to(device)

        train_ds = TensorDataset(X_train_fold, y_train_fold)
        val_ds = TensorDataset(X_val_fold, y_val_fold)
        train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

        model = LargeRNAModelVQC(n_qubits=n_qubits, q_layers=q_layers, device=device).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr_classical)
        criterion = nn.MSELoss()

        for _ in range(10):
            train_one_epoch(model, optimizer, criterion, train_loader, device)

        _, val_metrics = validate_model(model, criterion, val_loader, device)

        # ✅ Store all metrics per fold
        for key in metrics_all_folds.keys():
            metrics_all_folds[key].append(val_metrics[key])

    # ✅ Compute mean & std across folds
    final_metrics = {key: (np.mean(metrics_all_folds[key]), np.std(metrics_all_folds[key]))
                     for key in metrics_all_folds}

    print(f"Mean/Std across 5 folds:")
    print(f"MAE={final_metrics['mae'][0]:.4f}±{final_metrics['mae'][1]:.4f}, "
          f"MSE={final_metrics['mse'][0]:.4f}±{final_metrics['mse'][1]:.4f}, "
          f"RMSE={final_metrics['rmse'][0]:.4f}±{final_metrics['rmse'][1]:.4f}, "
          f"R2={final_metrics['r2'][0]:.4f}±{final_metrics['r2'][1]:.4f}")

    return final_metrics['r2'][0]  # ✅ Optimize using mean R² across folds


# ----------------------------------------------------------------------------
# Main Model Training with Best Hyperparams
# ----------------------------------------------------------------------------

def train_large_model_vqc_with_optuna(X_train, y_train, X_val, y_val, device, epochs=500):
    X_train, y_train = np.array(X_train), np.array(y_train)
    X_val, y_val = np.array(X_val), np.array(y_val)

    # ✅ Run Optuna Hyperparameter Search
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial), n_trials=10)
    best_params = study.best_params
    print(f"Best Hyperparams Found: {best_params}")

    model = LargeRNAModelVQC(n_qubits=best_params["n_qubits"], q_layers=best_params["q_layers"], device=device).to(device)
    optimizer = optim.Adam(model.parameters(), lr=best_params["lr_classical"])
    criterion = nn.MSELoss()

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32).to(device),
                             torch.tensor(y_train, dtype=torch.float32).to(device))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32).to(device),
                           torch.tensor(y_val, dtype=torch.float32).to(device))

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

    # ✅ Start timer for efficiency measurement
    start_time = time.time()

    # ✅ Initialize metrics history (moved outside the loop)
    train_metrics_history = {'mae': [], 'mse': [], 'rmse': [], 'r2': []}

    # ✅ Training loop
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
        val_loss, val_metrics = validate_model(model, criterion, val_loader, device)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val R²: {val_metrics['r2']:.4f}")

        # ✅ Store validation metrics per epoch
        for key in train_metrics_history.keys():
            train_metrics_history[key].append(val_metrics[key])

        # ✅ Measure GPU Memory (if using CUDA)
        if device == "cuda":
            allocated_mem = torch.cuda.memory_allocated(device) / (1024 ** 2)  # Convert to MB
            print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val R²: {val_metrics['r2']:.4f} | GPU Memory: {allocated_mem:.2f} MB")

    # ✅ End timer
    end_time = time.time()
    total_time = end_time - start_time

    # ✅ Compute Final Mean/Std for the Full Training Process
    final_metrics = {key: (np.mean(train_metrics_history[key]), np.std(train_metrics_history[key]))
                     for key in train_metrics_history}

    print(f"\nFinal Mean/Std over {epochs} epochs:")
    print(f"MAE={final_metrics['mae'][0]:.4f}±{final_metrics['mae'][1]:.4f}, "
          f"MSE={final_metrics['mse'][0]:.4f}±{final_metrics['mse'][1]:.4f}, "
          f"RMSE={final_metrics['rmse'][0]:.4f}±{final_metrics['rmse'][1]:.4f}, "
          f"R2={final_metrics['r2'][0]:.4f}±{final_metrics['r2'][1]:.4f}")

    print(f"Total Training Time: {total_time:.2f} seconds")

    return model





# ✅ Run Training
if __name__ == "__main__":
    #device = "cpu"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = train_large_model_vqc_with_optuna(X_train, y_train, X_val, y_val, device)


[I 2025-02-15 13:51:22,682] A new study created in memory with name: no-name-8a2cc7c4-df13-4a87-a945-77854fcf3461


Raw linear_in output: mean = -0.0310, std = 0.1425
Scaled linear_in output (tanh): mean = -0.6010, std = 2.8235


<ipython-input-18-4e2c49d62139>:69: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  exp_vals.append(torch.tensor(c.expectation_ps(z=[k]), device=self.device))
<ipython-input-18-4e2c49d62139>:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  exp_vals.append(torch.tensor(c.expectation_ps(x=[k]), device=self.device))


Quantum expectations: min = -0.3196, max = 0.6761
Final output (after linear_out): [0.01678567 0.01243923 0.01355328 0.01880601 0.01744062 0.0204152
 0.00931673 0.01587883]
Raw linear_in output: mean = -0.0308, std = 0.1414
Scaled linear_in output (tanh): mean = -0.5975, std = 2.8018
Quantum expectations: min = -0.3332, max = 0.6993
Final output (after linear_out): [0.01441075 0.02021679 0.0139234  0.01691195 0.01710926 0.0164063
 0.01612704 0.01224673]
Raw linear_in output: mean = -0.0306, std = 0.1416
Scaled linear_in output (tanh): mean = -0.5936, std = 2.8060
Quantum expectations: min = -0.3112, max = 0.6931
Final output (after linear_out): [0.01705441 0.01963058 0.01973839 0.01042009 0.01670966 0.01031235
 0.01552171 0.01679244]
Raw linear_in output: mean = -0.0309, std = 0.1412
Scaled linear_in output (tanh): mean = -0.6002, std = 2.7983
Quantum expectations: min = -0.3345, max = 0.6635
Final output (after linear_out): [0.01689091 0.0213257  0.01648513 0.01841051 0.02487741 0.017

[I 2025-02-15 14:47:42,940] Trial 0 finished with value: -5.187755870819092 and parameters: {'lr_classical': 5.814622013822599e-05, 'lr_quantum': 0.00034858635636169386, 'n_qubits': 12, 'q_layers': 9}. Best is trial 0 with value: -5.187755870819092.


Quantum expectations: min = -0.6410, max = 0.1851
Final output (after linear_out): [0.13390422 0.13573055]
Mean/Std across 5 folds:
MAE=37.1438±1.8523, MSE=1654.0028±164.1631, RMSE=40.6197±2.0105, R2=-5.1878±0.7037
Raw linear_in output: mean = -0.0071, std = 0.1162
Scaled linear_in output (tanh): mean = -0.1336, std = 2.3147
Quantum expectations: min = -0.8555, max = 0.7320
Final output (after linear_out): [0.02489088 0.03161587 0.02756125 0.03850023 0.02141921 0.03179082
 0.0252254  0.04022336]
Raw linear_in output: mean = -0.0072, std = 0.1158
Scaled linear_in output (tanh): mean = -0.1343, std = 2.3055
Quantum expectations: min = -0.8556, max = 0.7536
Final output (after linear_out): [0.03324579 0.03756952 0.04018107 0.0437786  0.03715847 0.03920465
 0.03387498 0.03918076]
Raw linear_in output: mean = -0.0078, std = 0.1160
Scaled linear_in output (tanh): mean = -0.1473, std = 2.3103
Quantum expectations: min = -0.8588, max = 0.7495
Final output (after linear_out): [0.04670931 0.0457

... (14 outputs left out)

[I 2025-02-15 18:23:44,187] Trial 8 finished with value: -5.234478855133057 and parameters: {'lr_classical': 1.1699107015917987e-05, 'lr_quantum': 0.006351177613607735, 'n_qubits': 8, 'q_layers': 8}. Best is trial 2 with value: -4.374240684509277.


Quantum expectations: min = -0.8182, max = 0.6646
Final output (after linear_out): [0.039726   0.04582403]
Mean/Std across 5 folds:
MAE=37.3191±1.8865, MSE=1667.1873±168.7140, RMSE=40.7796±2.0529, R2=-5.2345±0.6952
Raw linear_in output: mean = -0.0583, std = 0.0930
Scaled linear_in output (tanh): mean = -1.1697, std = 1.8657
Quantum expectations: min = -0.8231, max = 0.8671
Final output (after linear_out): [ 0.0115868   0.00916502  0.00502141 -0.00962621  0.00044304 -0.00468351
  0.0023758  -0.00964911]
Raw linear_in output: mean = -0.0578, std = 0.0931
Scaled linear_in output (tanh): mean = -1.1594, std = 1.8672
Quantum expectations: min = -0.8371, max = 0.8654
Final output (after linear_out): [ 1.5576929e-03  5.6004971e-03  1.0787398e-03  2.2634864e-05
 -2.8057247e-03 -1.3341188e-02 -7.5302422e-03 -1.4298603e-02]
Raw linear_in output: mean = -0.0582, std = 0.0933
Scaled linear_in output (tanh): mean = -1.1670, std = 1.8708
Quantum expectations: min = -0.8110, max = 0.8460
Final outpu

[I 2025-02-15 19:14:47,625] Trial 9 finished with value: -4.850768756866455 and parameters: {'lr_classical': 0.0003256222590061575, 'lr_quantum': 0.00014635509813337312, 'n_qubits': 12, 'q_layers': 8}. Best is trial 2 with value: -4.374240684509277.


Streaming output truncated to the last 5000 lines.
Final output (after linear_out): [36.611347 38.30663  39.174686 39.363842 38.561924 35.848625 38.951794
 36.58979 ]
Raw linear_in output: mean = 0.0560, std = 0.0961
Scaled linear_in output (tanh): mean = 1.1148, std = 1.9082
Quantum expectations: min = -0.8391, max = 0.8375
Final output (after linear_out): [37.953915 39.53324  32.12217  39.27387  36.240288 39.09672  36.342785
 37.50911 ]
Raw linear_in output: mean = 0.0560, std = 0.0961
Scaled linear_in output (tanh): mean = 1.1138, std = 1.9099
Quantum expectations: min = -0.8386, max = 0.8270
Final output (after linear_out): [33.084892 37.760674 37.824734 38.80874  40.162533 37.184505 38.241764
 36.70929 ]
Raw linear_in output: mean = 0.0560, std = 0.0963
Scaled linear_in output (tanh): mean = 1.1139, std = 1.9132
Quantum expectations: min = -0.8135, max = 0.8302
Final output (after linear_out): [36.621708 38.62253  38.951824 38.32949  39.059345 36.3543   36.65905
 36.209415]
Raw li